# Model 6 / 8 — MobileNetV2 — Sunflower Alternaria Blight Severity Classification

Six-class severity classifier (Immune, Resistant, MR, MS, S, HS — Mayee & Datar 1986 scale).
Transfer learning from ImageNet, class-imbalance handling, full performance + efficiency metrics.

**Run order:** Sections 1–2 must be run identically on all 8 systems (same `split.csv`, same seed).
Only Section 4 (model definition) changes between the 8 notebooks.

Lightweight backbone — this is the model the project doc flags as most relevant to the eventual
web/app deployment (Section 8), so its efficiency numbers (Section 10) matter as much as accuracy.

| Setting | Value |
|---|---|
| Model | MobileNetV2 (torchvision, ImageNet weights) |
| Input size | 224×224 |
| Early stopping patience | 25 (on val macro-F1) |
| Batch size | 32 |
| Optimizer | AdamW |
| LR schedule | Two-stage: frozen head warmup, then full fine-tune w/ cosine decay |


## 0. Setup

In [ ]:
!pip install -q timm thop scikit-learn matplotlib seaborn pandas tqdm
import os, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, roc_auc_score, cohen_kappa_score,
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
from thop import profile as thop_profile
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate partially-truncated JPEGs instead of crashing
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Dataset access (Google Drive)

Fill in **one** of the two options below.

- **Colab + Drive mount** (simplest if you're on Colab and the data is in your own Drive)
- **gdown** (works anywhere, incl. local/Kaggle, if the folder/file is shared "Anyone with the link")

Paste your actual link/ID once you have it — code is ready either way, just uncomment the block you need.


In [ ]:
# Dataset is already local -- no Drive download needed.
DATA_ROOT = "/home/sem_7th/Downloads/Sunflower"

# Expected layout (ImageFolder-style, one subfolder per class):
# DATA_ROOT/
#   Class 1/  *.jpg
#   Class 2/  *.jpg
#   ...
#   Class 6/  *.jpg

CLASS_NAMES = ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5", "Class 6"]
SEVERITY_SCORE = {0: 0, 1: 1, 2: 3, 3: 5, 4: 7, 5: 9}          # label idx -> 0-9 score
SEVERITY_LABEL = {0: "Immune", 1: "Resistant", 2: "MR", 3: "MS", 4: "S", 5: "HS"}

## 2. Stratified 70/15/15 split — GENERATE ONCE, REUSE ON ALL 8 SYSTEMS

Run this cell only the **first** time. It writes `split.csv` (path, label, split).
Copy that exact file to the other 7 systems/notebooks and load it instead of
regenerating — this guarantees every one of the 8 models is trained/evaluated
on identical data, which is required for the comparison in Section 4 of the report.


In [ ]:
SPLIT_CSV = "split.csv"


def is_valid_image_file(path):
    try:
        from PIL import Image
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


if not os.path.exists(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}. Update it to your Google Drive folder path.")

if not os.path.exists(SPLIT_CSV):
    records = []
    missing_dirs = []
    skipped = 0
    for label_idx, cname in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATA_ROOT, cname)
        if not os.path.isdir(class_dir):
            missing_dirs.append(class_dir)
            continue
        for fname in sorted(os.listdir(class_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            fpath = os.path.join(class_dir, fname)
            if not is_valid_image_file(fpath):
                skipped += 1
                print(f"Skipping unreadable image: {fpath}")
                continue
            records.append({"path": fpath, "label": label_idx})

    if not records:
        raise FileNotFoundError(
            "No valid images were found. Check that DATA_ROOT and CLASS_NAMES match your Drive folder structure. "
            f"Expected folders: {[os.path.join(DATA_ROOT, c) for c in CLASS_NAMES]}"
        )

    if missing_dirs:
        print("Missing class folders:")
        for d in missing_dirs:
            print(" -", d)
    if skipped:
        print(f"Skipped unreadable files: {skipped}")

    df = pd.DataFrame(records)
    print("Total valid images found:", len(df))
    print(df["label"].value_counts().sort_index())

    train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
    val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df["label"], random_state=SEED)

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    full = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
    full.to_csv(SPLIT_CSV, index=False)
    print(f"Saved {SPLIT_CSV} -- COPY THIS FILE TO ALL 8 SYSTEMS AND REUSE IT.")
else:
    print(f"{SPLIT_CSV} already exists -- loading existing split (do NOT regenerate).")

split_df = pd.read_csv(SPLIT_CSV)
valid_mask = split_df["path"].apply(is_valid_image_file)
split_df = split_df[valid_mask].reset_index(drop=True)
print(f"Kept {len(split_df)} valid rows after filtering unreadable files.")
print(split_df.groupby(["split", "label"]).size().unstack(fill_value=0))


## 3. Dataset / DataLoaders + class-imbalance handling

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Phone-camera-aware augmentation for the TRAIN set only
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),        # leaves have no fixed "up" -- biologically fine
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),  # phone motion/focus blur
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class LeafDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].tolist()
        self.labels = df["label"].tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        from PIL import Image
        path = self.paths[idx]
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                img = self.transform(img)
            return img, self.labels[idx]
        except Exception as e:
            raise ValueError(f"Unreadable image file: {path}") from e

train_ds = LeafDataset(split_df[split_df.split == "train"], train_tf)
val_ds   = LeafDataset(split_df[split_df.split == "val"], eval_tf)
test_ds  = LeafDataset(split_df[split_df.split == "test"], eval_tf)

# --- class imbalance: balanced batch sampling for train loader ---
train_labels = split_df[split_df.split == "train"]["label"].values
class_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("Train class counts:", class_counts)
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

# Also compute per-class weights for an (optional) weighted loss term.
# We use the sampler as the primary imbalance fix and keep the loss unweighted
# + light label smoothing, to avoid double-correcting. Flip WEIGHTED_LOSS=True
# to additionally weight CE if the sampler alone isn't enough on your data.
WEIGHTED_LOSS = False
loss_class_weights = torch.tensor(
    class_weights / class_weights.sum() * len(CLASS_NAMES),
    dtype=torch.float32
).to(DEVICE)

## 4. Model — MobileNetV2 (transfer learning)

In [ ]:
MODEL_NAME = "MobileNetV2"

def build_model(num_classes=6):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
    # m.classifier is Sequential(Dropout(p=0.2), Linear(1280, 1000)) -- replace the whole
    # thing so the dropout rate matches the other 7 notebooks (0.3) for a fair comparison.
    in_feats = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_feats, num_classes)
    )
    return m

model = build_model(len(CLASS_NAMES)).to(DEVICE)

def set_backbone_trainable(model, trainable: bool):
    for name, p in model.named_parameters():
        if "classifier" not in name:
            p.requires_grad = trainable


## 5. Training config

**Two-stage fine-tuning:**
1. Freeze backbone, train the new head only (fast, stabilizes the randomly-initialized classifier).
2. Unfreeze everything, fine-tune end-to-end at a much lower LR.

Early stopping (patience = 25, monitored on **validation macro-F1**, not just loss/accuracy,
since macro-F1 is what actually reflects performance on the minority classes) applies to stage 2.


In [ ]:
STAGE1_EPOCHS = 10          # head-only warmup
STAGE2_MAX_EPOCHS = 150     # capped upper bound; early stopping will normally trigger sooner
PATIENCE = 25
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05


criterion = nn.CrossEntropyLoss(
    weight=loss_class_weights if WEIGHTED_LOSS else None,
    label_smoothing=LABEL_SMOOTHING
)

CHECKPOINT_PATH = f"best_{MODEL_NAME}.pth"


## 6. Train / eval loop helpers

In [ ]:
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train_mode):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_preds.append(outputs.argmax(1).detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_preds = np.concatenate(all_preds); all_labels = np.concatenate(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)[2]
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, macro_f1


def train_stage(model, optimizer, scheduler, n_epochs, patience, history, start_epoch=0):
    best_f1 = -1
    epochs_no_improve = 0
    for epoch in range(start_epoch, start_epoch + n_epochs):
        t0 = time.time()
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, optimizer=None)
        epoch_time = time.time() - t0
        if scheduler is not None:
            scheduler.step(val_f1)

        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc, train_f1=train_f1,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1, epoch_time_s=epoch_time))
        print(f"[{epoch:03d}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} | {epoch_time:.1f}s")

        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {patience} epochs).")
                break
    return epoch + 1  # last epoch index reached


## 7. Run training — Stage 1 (head only) → Stage 2 (full fine-tune, early stopping patience=25)

In [ ]:
history = []

# --- Stage 1: frozen backbone, head warmup ---
set_backbone_trainable(model, trainable=False)
optimizer1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=LR_STAGE1, weight_decay=WEIGHT_DECAY)
print("=== Stage 1: training head only ===")
last_epoch = train_stage(model, optimizer1, scheduler=None, n_epochs=STAGE1_EPOCHS,
                          patience=STAGE1_EPOCHS + 1, history=history, start_epoch=0)

# --- Stage 2: unfreeze everything, fine-tune with early stopping ---
set_backbone_trainable(model, trainable=True)
optimizer2 = torch.optim.AdamW(model.parameters(), lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode="max", factor=0.5, patience=8)
print("=== Stage 2: full fine-tune (early stopping patience=25 on val macro-F1) ===")
train_stage(model, optimizer2, scheduler2, n_epochs=STAGE2_MAX_EPOCHS, patience=PATIENCE,
            history=history, start_epoch=last_epoch)

# reload best checkpoint before evaluation
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
print("Loaded best checkpoint for final evaluation.")


@torch.no_grad()
def evaluate_on_split(model, loader, name):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    print(f"{name.upper()} ACC={acc:.4f} | Macro-P={prec_macro:.4f} | Macro-R={rec_macro:.4f} | Macro-F1={f1_macro:.4f}")
    return acc, f1_macro


evaluate_on_split(model, test_loader, "test")


## 8. Learning curves

In [ ]:
hist_df = pd.DataFrame(history)
hist_df.to_csv(f"history_{MODEL_NAME}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df.epoch, hist_df.train_loss, label="train")
axes[0].plot(hist_df.epoch, hist_df.val_loss, label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(hist_df.epoch, hist_df.train_f1, label="train")
axes[1].plot(hist_df.epoch, hist_df.val_f1, label="val")
axes[1].set_title("Macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"learning_curves_{MODEL_NAME}.png", dpi=150)
plt.show()


## 9. Test-set performance metrics (Section 3 of the report)

In [ ]:
@torch.no_grad()
def get_test_predictions(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)

logits, y_true = get_test_predictions(model, test_loader)
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
y_pred = probs.argmax(1)

acc = accuracy_score(y_true, y_pred)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")  # ordinal severity -> quadratic weighting

try:
    auc_ovr = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
except ValueError:
    auc_ovr = float("nan")

cm = confusion_matrix(y_true, y_pred)
# per-class specificity from the confusion matrix
specificities = []
for i in range(len(CLASS_NAMES)):
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    specificities.append(tn / (tn + fp + 1e-9))
specificity_macro = float(np.mean(specificities))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
print(f"Accuracy: {acc:.4f}")
print(f"Macro  P/R/F1: {prec_macro:.4f} / {rec_macro:.4f} / {f1_macro:.4f}")
print(f"Weighted P/R/F1: {prec_w:.4f} / {rec_w:.4f} / {f1_w:.4f}")
print(f"Specificity (macro): {specificity_macro:.4f}")
print(f"AUC-ROC (OvR, macro): {auc_ovr:.4f}")
print(f"Cohen's kappa (quadratic-weighted): {kappa:.4f}")

# confusion matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{MODEL_NAME} -- Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{MODEL_NAME}.png", dpi=150)
plt.show()


## 10. Efficiency metrics — params, model size, FLOPs, training time, inference latency

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.save(model.state_dict(), f"final_{MODEL_NAME}.pth")
model_size_mb = os.path.getsize(f"final_{MODEL_NAME}.pth") / (1024 ** 2)

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
flops, _ = thop_profile(model, inputs=(dummy_input,), verbose=False)
gflops = flops / 1e9

avg_epoch_time_s = hist_df["epoch_time_s"].mean()

# --- inference latency: batch=1, average of 100 runs after 10 warmup runs ---
def measure_latency(model, device, n_warmup=10, n_runs=100):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_runs):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.time()
    return (t1 - t0) / n_runs * 1000  # ms/image

latency_gpu_ms = measure_latency(model, DEVICE) if DEVICE.type == "cuda" else None
latency_cpu_ms = measure_latency(model.to("cpu"), torch.device("cpu"))
model.to(DEVICE)  # move back

print(f"Params (total / trainable): {num_params:,} / {num_params_trainable:,}")
print(f"Model size (.pth): {model_size_mb:.2f} MB")
print(f"FLOPs: {gflops:.2f} GFLOPs")
print(f"Avg training time/epoch: {avg_epoch_time_s:.1f} s")
print(f"Inference latency -- GPU: {latency_gpu_ms} ms/img | CPU: {latency_cpu_ms:.2f} ms/img")


## 11. Save this model's results row

This appends one row to `results_master.csv`. Once all 8 models (run across your 8 systems)
have appended their row to a **shared copy** of this file, that CSV *is* the Section 4
comparison table — feed it straight into the trade-off plots (accuracy/F1 vs. latency vs. size).


In [ ]:
result_row = {
    "model": MODEL_NAME,
    "accuracy": acc,
    "precision_macro": prec_macro,
    "recall_macro": rec_macro,
    "f1_macro": f1_macro,
    "precision_weighted": prec_w,
    "recall_weighted": rec_w,
    "f1_weighted": f1_w,
    "specificity_macro": specificity_macro,
    "auc_roc_ovr_macro": auc_ovr,
    "cohen_kappa_quadratic": kappa,
    "params_total": num_params,
    "params_trainable_stage2": num_params_trainable,
    "model_size_mb": model_size_mb,
    "gflops": gflops,
    "avg_train_time_per_epoch_s": avg_epoch_time_s,
    "epochs_run": int(hist_df.epoch.max()) + 1,
    "inference_latency_gpu_ms": latency_gpu_ms,
    "inference_latency_cpu_ms": latency_cpu_ms,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

MASTER_CSV = "results_master.csv"
row_df = pd.DataFrame([result_row])
if os.path.exists(MASTER_CSV):
    row_df.to_csv(MASTER_CSV, mode="a", header=False, index=False)
else:
    row_df.to_csv(MASTER_CSV, index=False)

print(row_df.T)
print(f"\nAppended to {MASTER_CSV}. Copy/merge this file across your 8 systems to build the full comparison table.")


---
### Notes for the other 7 models

Only **Section 4** (`build_model`) and `MODEL_NAME` need to change per system — everything else
(split, augmentation, imbalance handling, training loop, metrics, efficiency measurement, results
schema) should stay identical so the final 8-row `results_master.csv` is a fair comparison.
Section 0's `ImageFile.LOAD_TRUNCATED_IMAGES = True` and Section 2's per-image validation before
writing `split.csv` are carried over from the fixes found while training models 1 and 2 -- keep both
in every remaining notebook. InceptionV3 (model 3) needed `IMG_SIZE = 299` and a tuple-output check
in `run_epoch`; that check is harmless here (MobileNet returns a plain tensor) and isn't included in
this notebook since this fixed baseline was branched before that change -- add it back if you're
running this on the same system as model 3 and want Section 6 to be byte-identical everywhere.
Once all 8 are done, the next step is the accuracy/F1-vs-efficiency trade-off plots and best-model
selection (Section 4 of the project doc), then the Grad-CAM/XAI suite (Section 5) on the winner.
